# Week 6: Data Retrieval and Processing
This notebook implements the data retrieval and preprocessing pipeline for Milestone 1 IoT data stored on the blockchain, as specified in the [docs/Code Template.md](file:///Users/brianjancarlos/codestuff/MMDC/ADET/adet/docs/Code%20Template.md).

Reference: [MS1_Smart_Tracking_System_Blockchain_Ledger_Submission_TeamKaizen.ipynb](file:///Users/brianjancarlos/codestuff/MMDC/ADET/adet/MS1_Smart_Tracking_System_Blockchain_Ledger_Submission_TeamKaizen.ipynb)

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from web3 import Web3

def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell environment variables, fallback to local .env
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default

def get_env_int(key, default):
    return int(get_env_value(key, str(default)))

def get_env_float(key, default):
    return float(get_env_value(key, str(default)))

# Connect to local Ganache blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [2]:
# Load deployed smart contract configuration
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load ABI
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Instantiate the contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Configure default account
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if override_owner:
        contract_owner = Web3.to_checksum_address(override_owner)
    else:
        contract_owner = web3.eth.accounts[0]

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using default sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using default sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [3]:
# Get the total number of stored records from blockchain
total_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {total_records}")

# Retrieve and print the first stored record to verify retrieval works
if total_records > 0:
    first_record = contract.functions.getRecord(0).call()
    print("First Stored Record:", first_record)
else:
    print("No records stored yet on the blockchain.")

Total IoT records stored: 166
First Stored Record: [1780192485, 'PKG7545', 'Location', 'Naha Central Post Office']


In [4]:
# Fetch all stored IoT data and structure it in a DataFrame
data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

# Convert to a DataFrame
df = pd.DataFrame(data)

# Convert timestamp to readable format
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

# Display first few records
print("Raw retrieved blockchain data preview:")
display(df.head())

Raw retrieved blockchain data preview:


,timestamp,device_id,data_type,data_value
0,2026-05-31 01:54:45,PKG7545,Location,Naha Central Post Office
1,2026-05-31 01:54:45,PKG7545,Status,Out for Delivery
2,2026-05-31 01:54:46,PKG2659,Location,Nagoya Central Post Office
3,2026-05-31 01:54:46,PKG2659,Status,Arrival
4,2026-05-31 01:54:46,PKG7965,Location,Nagoya Central Post Office


In [5]:
# Data Preprocessing and Cleaning
print("Identifying missing values before cleaning:")
print(df.isna().sum())

# Extract numerical values from 'data_value'
# Note: Improved regex r'(-?\d+\.?\d*)' is used instead of template r'(\d+\.?\d*)' 
# to correctly capture negative numbers such as negative temperatures.
df["numeric_value"] = df["data_value"].str.extract(r'(-?\d+\.?\d*)').astype(float)

# Handle missing values (if any)
# Fill missing numeric values with the median of the column (if significant) or 0 (if minor)
# The template suggests using fillna(0) for minor missing values.
df.fillna(0, inplace=True)

# Display cleaned data
print("\nCleaned and preprocessed data preview:")
display(df.head())

Identifying missing values before cleaning:
timestamp     0
device_id     0
data_type     0
data_value    0
dtype: int64

Cleaned and preprocessed data preview:


,timestamp,device_id,data_type,data_value,numeric_value
0,2026-05-31 01:54:45,PKG7545,Location,Naha Central Post Office,0.0
1,2026-05-31 01:54:45,PKG7545,Status,Out for Delivery,0.0
2,2026-05-31 01:54:46,PKG2659,Location,Nagoya Central Post Office,0.0
3,2026-05-31 01:54:46,PKG2659,Status,Arrival,0.0
4,2026-05-31 01:54:46,PKG7965,Location,Nagoya Central Post Office,0.0


In [6]:
# Save cleaned IoT data to a CSV file in the assets/ directory
output_path = "assets/cleaned_iot_data.csv"
df.to_csv(output_path, index=False)

print(f"✅ Cleaned IoT data saved successfully as {output_path}")

✅ Cleaned IoT data saved successfully as assets/cleaned_iot_data.csv
